# Notebook 05 — Embeddings y Búsqueda Semántica

**Semana:** 5  
**Bloque:** 4.1 — Embeddings y modelos de embeddings  
**Tiempo estimado:** 3-3.5 horas

## Qué vas a construir

- Similitud coseno implementada desde cero con numpy
- Buscador semántico sobre un corpus de 80 frases sin ningún framework
- Visualización 2D de embeddings con PCA para ver los clusters semánticos
- Comparativa directa: búsqueda semántica vs búsqueda por palabras clave sobre las mismas queries
- **Reto final:** FAQ semántica — dado un banco de preguntas frecuentes, encuentra la más parecida a la query del usuario

## Prerrequisitos

```bash
pip install openai numpy matplotlib scikit-learn python-dotenv
# Opcional para UMAP:
pip install umap-learn
```

> **Nota sobre costes:** este notebook hace llamadas a la API de embeddings de OpenAI.
> Con `text-embedding-3-small` y los corpus de este notebook (~5.000 tokens en total)
> el coste es menor de **$0.001**. Completamente insignificante.

---
## Celda 0 — Setup

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
cliente = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODELO_EMBEDDING = "text-embedding-3-small"

# ── Función base: genera embedding de uno o varios textos ─────────────────────
def generar_embedding(texto: str) -> np.ndarray:
    """Genera el embedding de un texto. Devuelve np.ndarray de shape (1536,)."""
    respuesta = cliente.embeddings.create(
        model=MODELO_EMBEDDING,
        input=texto
    )
    return np.array(respuesta.data[0].embedding)


def generar_embeddings_batch(textos: list[str]) -> np.ndarray:
    """
    Genera embeddings de una lista de textos en una sola llamada API.
    Devuelve np.ndarray de shape (n_textos, 1536).
    Más eficiente que llamar a generar_embedding() en un bucle.
    """
    respuesta = cliente.embeddings.create(
        model=MODELO_EMBEDDING,
        input=textos
    )
    # Los resultados vienen ordenados por índice
    vectores = [item.embedding for item in sorted(respuesta.data, key=lambda x: x.index)]
    return np.array(vectores)


# Prueba rápida
v_prueba = generar_embedding("Hola mundo")
print(f"Dimensiones del embedding: {v_prueba.shape}")
print(f"Tipo de datos:             {v_prueba.dtype}")
print(f"Norma del vector:          {np.linalg.norm(v_prueba):.4f}  (debería ser ~1.0)")

---
## Celda 1 — Similitud coseno desde cero

Implementa la similitud coseno sin usar ninguna librería de ML.
Luego úsala para explorar qué pares de frases el modelo considera similares.

**Antes de ejecutar:** mira las frases del grupo de prueba y predice
cuáles tendrán similitud > 0.85 y cuáles < 0.3.

In [ ]:
# ── Implementación desde cero ─────────────────────────────────────────────────

def similitud_coseno(a: np.ndarray, b: np.ndarray) -> float:
    """Similitud coseno entre dos vectores. Rango: [-1, 1]."""
    norma_a = np.linalg.norm(a)
    norma_b = np.linalg.norm(b)
    if norma_a == 0 or norma_b == 0:
        return 0.0
    return float(np.dot(a, b) / (norma_a * norma_b))


def similitudes_batch(query: np.ndarray, corpus: np.ndarray) -> np.ndarray:
    """
    Similitud entre un vector query y todos los vectores del corpus.
    Usa operaciones matriciales — mucho más rápido que un bucle.

    query:  shape (d,)
    corpus: shape (n, d)
    return: shape (n,)  — una similitud por documento
    """
    query_norm  = query  / np.linalg.norm(query)
    normas      = np.linalg.norm(corpus, axis=1, keepdims=True)  # shape (n, 1)
    corpus_norm = corpus / normas
    return corpus_norm @ query_norm   # producto matricial: (n, d) @ (d,) = (n,)


# ── Grupos de frases para experimentar ───────────────────────────────────────
# 4 grupos temáticos + 2 frases anómalas

FRASES = [
    # Grupo A: perros
    "El perro ladra en el jardín",
    "El can aúlla por la noche",
    "Mi mascota canina come pienso",
    "Los canes son animales domésticos leales",

    # Grupo B: programación Python
    "Python es un lenguaje de programación interpretado",
    "Las listas en Python son mutables",
    "El intérprete de CPython ejecuta bytecode",
    "Los decoradores son una característica avanzada de Python",

    # Grupo C: cocina
    "La paella valenciana lleva arroz y azafrán",
    "El sofrito es la base de muchas recetas mediterráneas",
    "Precalienta el horno a 180 grados antes de hornear",
    "El aceite de oliva virgen extra es el mejor para freír",

    # Grupo D: inteligencia artificial
    "Los modelos de lenguaje generan texto token a token",
    "El fine-tuning adapta un modelo preentrenado a una tarea específica",
    "Los embeddings representan el significado semántico en vectores",
    "La atención multi-cabeza es el mecanismo central del Transformer",

    # Frases anómalas (sin grupo claro)
    "El número 42 es la respuesta a todo",
    "Mañana lloverá en el norte de la península",
]

ETIQUETAS = (
    ["🐕 Perros"] * 4 +
    ["🐍 Python"] * 4 +
    ["🍳 Cocina"] * 4 +
    ["🤖 IA"]     * 4 +
    ["❓ Otro"]   * 2
)

print(f"Generando {len(FRASES)} embeddings...")
MATRIZ = generar_embeddings_batch(FRASES)   # shape (18, 1536)
print(f"Matriz de embeddings: {MATRIZ.shape}")


# ── Experimento 1: similitudes entre pares específicos ────────────────────────
print("\n=== Similitudes entre pares ===")
pares = [
    (0, 1, "perro ↔ can"),
    (0, 4, "perro ↔ Python"),
    (4, 5, "Python interpretado ↔ listas Python"),
    (12, 14, "LLM tokens ↔ embeddings"),
    (8, 12, "paella ↔ LLM tokens"),
    (0, 3, "perro ladra ↔ canes domésticos"),
]
for i, j, descripcion in pares:
    sim = similitud_coseno(MATRIZ[i], MATRIZ[j])
    barra = "█" * int(sim * 20)
    print(f"  {descripcion:<40} {sim:.4f}  {barra}")

**Observaciones** (edita esta celda):

- El par con mayor similitud fue: ... con score ...
- El par con menor similitud fue: ... con score ...
- ¿Sorprendió algún resultado? ¿Cuál y por qué?
- ¿A partir de qué umbral dirías que dos frases son "relacionadas"?

---
## Celda 2 — Buscador semántico sobre corpus de 80 frases

Amplía el corpus a 80 frases para que la búsqueda sea más interesante.
El buscador no usa ninguna base de datos: solo numpy.
Es exactamente lo que ocurre dentro de una vector DB, pero visible.

In [ ]:
# ── Corpus ampliado: 80 frases en 8 categorías ───────────────────────────────

CORPUS = {
    "machine_learning": [
        "El aprendizaje supervisado requiere datos etiquetados",
        "El overfitting ocurre cuando el modelo memoriza el conjunto de entrenamiento",
        "La validación cruzada evalúa la generalización del modelo",
        "Los árboles de decisión particionan el espacio de características",
        "El gradient boosting combina múltiples modelos débiles",
        "La regularización L2 penaliza los pesos grandes del modelo",
        "La normalización de características acelera la convergencia",
        "Las métricas de evaluación dependen del tipo de problema",
        "El aprendizaje por refuerzo optimiza una política mediante recompensas",
        "Las redes neuronales aprenden representaciones jerárquicas",
    ],
    "llm_rag": [
        "Los modelos de lenguaje generan texto token a token",
        "RAG combina recuperación de documentos con generación de texto",
        "Los embeddings representan el significado semántico en vectores",
        "La ventana de contexto limita cuánto texto puede procesar el modelo",
        "El chunking divide documentos en fragmentos para indexarlos",
        "La búsqueda vectorial encuentra documentos por similitud semántica",
        "El prompt engineering mejora la calidad de las respuestas del LLM",
        "El fine-tuning adapta un modelo preentrenado a una tarea específica",
        "Los hallucinations son respuestas plausibles pero incorrectas",
        "La temperatura controla la aleatoriedad de las respuestas generadas",
    ],
    "python_programacion": [
        "Python es un lenguaje interpretado de tipado dinámico",
        "Los generadores en Python producen valores de forma diferida",
        "Los decoradores modifican el comportamiento de funciones y clases",
        "Los context managers gestionan recursos con el protocolo with",
        "La comprensión de listas es más pythonico que los bucles for",
        "Los tipos hint mejoran la legibilidad y el tooling del código",
        "asyncio permite concurrencia mediante coroutines en Python",
        "Los entornos virtuales aíslan las dependencias de cada proyecto",
        "FastAPI usa Pydantic para validar los datos de entrada automáticamente",
        "Los dataclasses reducen el código boilerplate para clases de datos",
    ],
    "cloud_devops": [
        "Docker empaqueta aplicaciones con todas sus dependencias",
        "Kubernetes orquesta contenedores en clústeres de producción",
        "CI/CD automatiza el proceso de prueba y despliegue del código",
        "Las variables de entorno almacenan configuración sensible de forma segura",
        "Los microservicios descomponen aplicaciones en servicios independientes",
        "La infraestructura como código permite reproducir entornos exactamente",
        "Los health checks verifican que un servicio responde correctamente",
        "El escalado horizontal añade más instancias en lugar de más potencia",
        "Los logs estructurados facilitan el análisis y las alertas automáticas",
        "El rate limiting protege los servicios de picos de tráfico inesperados",
    ],
    "bases_datos": [
        "Las transacciones ACID garantizan consistencia en bases de datos relacionales",
        "Los índices aceleran las consultas al coste de más espacio en disco",
        "El modelo relacional organiza datos en tablas con claves foráneas",
        "NoSQL sacrifica consistencia por escalabilidad horizontal",
        "PostgreSQL soporta tipos de datos avanzados como JSONB y arrays",
        "Las bases de datos vectoriales indexan embeddings para búsqueda rápida",
        "El sharding distribuye datos entre múltiples nodos de la base de datos",
        "Las migraciones de esquema deben ser reversibles en producción",
        "Las consultas N+1 son un antipatrón común en ORMs mal configurados",
        "El connection pooling reutiliza conexiones para mejorar el rendimiento",
    ],
    "seguridad": [
        "La inyección SQL es una de las vulnerabilidades más peligrosas en aplicaciones web",
        "JWT permite transmitir información verificada entre servicios de forma segura",
        "El cifrado de extremo a extremo protege los datos en tránsito y reposo",
        "El principio de mínimo privilegio limita el daño ante una brecha de seguridad",
        "La autenticación multifactor reduce significativamente el riesgo de acceso no autorizado",
        "Las claves API nunca deben hardcodearse en el código fuente",
        "El prompt injection es un ataque específico de aplicaciones con LLMs",
        "RGPD requiere base legal para tratar datos personales de ciudadanos europeos",
        "Las dependencias desactualizadas son un vector de ataque frecuente",
        "El análisis estático de código detecta vulnerabilidades antes del despliegue",
    ],
    "negocio_freelance": [
        "El nicho de mercado define a quién ayudas y con qué problema específico",
        "La propuesta de valor articula por qué elegirte a ti frente a la competencia",
        "El precio por valor se basa en el resultado para el cliente, no en las horas",
        "El scoping del proyecto evita el scope creep y los malentendidos posteriores",
        "Un portfolio con demos desplegadas convierte mejor que un repositorio vacío",
        "El retainer mensual proporciona ingresos predecibles al freelance",
        "LinkedIn es el canal principal de captación B2B para servicios técnicos",
        "El contrato debe especificar la propiedad intelectual del código entregado",
        "La facturación de autónomo requiere incluir IVA e IRPF correctamente",
        "El NDA protege la información confidencial del cliente durante el proyecto",
    ],
    "medicina_salud": [
        "Los ensayos clínicos aleatorizados son el estándar de oro en investigación médica",
        "La inteligencia artificial está transformando el diagnóstico por imagen médica",
        "La farmacocinética estudia cómo el organismo procesa un medicamento",
        "Los biomarcadores permiten detectar enfermedades en estadios tempranos",
        "La telemedicina amplía el acceso a la atención sanitaria en zonas rurales",
        "El microbioma intestinal influye en el sistema inmunológico y el estado de ánimo",
        "La terapia génica permite corregir mutaciones causantes de enfermedades",
        "Los datos de salud están sujetos a regulaciones especiales de privacidad",
        "La resistencia antibiótica es una amenaza global de salud pública emergente",
        "La medicina personalizada adapta el tratamiento al perfil genético del paciente",
    ],
}

# Aplana el corpus en listas paralelas
TEXTOS_CORPUS   = []
CATEGORIAS_CORPUS = []
for categoria, frases in CORPUS.items():
    TEXTOS_CORPUS.extend(frases)
    CATEGORIAS_CORPUS.extend([categoria] * len(frases))

print(f"Generando embeddings para {len(TEXTOS_CORPUS)} documentos...")
EMBEDDINGS_CORPUS = generar_embeddings_batch(TEXTOS_CORPUS)   # shape (80, 1536)
print(f"Corpus indexado: {EMBEDDINGS_CORPUS.shape}")


# ── Buscador semántico ────────────────────────────────────────────────────────

def buscar_semantico(
    query: str,
    top_k: int = 5,
    umbral: float = 0.0
) -> list[dict]:
    """
    Busca los documentos más similares semánticamente a la query.
    
    Flujo:
    1. Genera el embedding de la query
    2. Calcula similitud coseno contra todos los documentos del corpus
    3. Devuelve los top_k más similares con score >= umbral
    """
    vector_query = generar_embedding(query)
    scores       = similitudes_batch(vector_query, EMBEDDINGS_CORPUS)
    indices_top  = np.argsort(scores)[::-1][:top_k]   # ordena descendente

    resultados = []
    for idx in indices_top:
        if scores[idx] >= umbral:
            resultados.append({
                "score":      float(scores[idx]),
                "texto":      TEXTOS_CORPUS[idx],
                "categoria":  CATEGORIAS_CORPUS[idx],
                "indice":     int(idx)
            })
    return resultados


def mostrar_resultados(query: str, resultados: list[dict]) -> None:
    print(f"\nQuery: '{query}'")
    print("-" * 70)
    for i, r in enumerate(resultados, 1):
        barra = "█" * int(r["score"] * 30)
        print(f"  {i}. [{r['score']:.4f}] {barra}")
        print(f"     {r['texto']}")
        print(f"     Categoría: {r['categoria']}")


# ── Pruebas del buscador ──────────────────────────────────────────────────────

QUERIES_PRUEBA = [
    "¿Cómo evitar que mi modelo memorice los datos de entrenamiento?",
    "bases de datos para guardar vectores",
    "cobrar más por mis servicios de consultoría",
    "proteger claves secretas en el código",
    "cáncer y tratamiento genético",              # término médico
    "async await concurrencia",                    # mezcla inglés/español
]

for query in QUERIES_PRUEBA:
    resultados = buscar_semantico(query, top_k=3)
    mostrar_resultados(query, resultados)

**Observaciones** (edita esta celda):

- Query con resultados más coherentes: ...
- Query con resultados más sorprendentes: ...
- ¿Los resultados cruzan categorías cuando tienen sentido? ¿Un ejemplo?
- ¿Qué umbral de score usarías para filtrar resultados irrelevantes?

---
## Celda 3 — Visualización 2D de los embeddings con PCA

Reduce 1536 dimensiones a 2 para visualizar los clusters semánticos.
Si los embeddings funcionan bien, los 8 grupos deben aparecer agrupados.

In [ ]:
# ── Reducción PCA ─────────────────────────────────────────────────────────────

pca = PCA(n_components=2, random_state=42)
embeddings_2d = pca.fit_transform(EMBEDDINGS_CORPUS)   # shape (80, 2)

varianza = pca.explained_variance_ratio_
print(f"Varianza explicada por PC1: {varianza[0]:.1%}")
print(f"Varianza explicada por PC2: {varianza[1]:.1%}")
print(f"Total conservada:           {varianza.sum():.1%}")
print(f"(El {1 - varianza.sum():.1%} restante se pierde en la reducción)")


# ── Gráfico de dispersión ─────────────────────────────────────────────────────

CATEGORIAS_UNICAS = list(CORPUS.keys())
COLORES           = cm.Set1(np.linspace(0, 1, len(CATEGORIAS_UNICAS)))
ICONOS = {
    "machine_learning":    "🤖",
    "llm_rag":             "💬",
    "python_programacion": "🐍",
    "cloud_devops":        "☁️",
    "bases_datos":         "🗄️",
    "seguridad":           "🔐",
    "negocio_freelance":   "💼",
    "medicina_salud":      "🏥",
}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Panel izquierdo: puntos con color por categoría ──────────────────────────
ax = axes[0]
for i, categoria in enumerate(CATEGORIAS_UNICAS):
    mask = np.array(CATEGORIAS_CORPUS) == categoria
    puntos = embeddings_2d[mask]
    ax.scatter(puntos[:, 0], puntos[:, 1],
               c=[COLORES[i]], label=f"{ICONOS[categoria]} {categoria}",
               s=80, alpha=0.8, edgecolors="white", linewidth=0.5)

ax.set_title(f"Clusters semánticos (PCA 2D)\nVarianza explicada: {varianza.sum():.1%}",
             fontsize=12)
ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
ax.set_xlabel(f"PC1 ({varianza[0]:.1%} var.)")
ax.set_ylabel(f"PC2 ({varianza[1]:.1%} var.)")
ax.grid(True, alpha=0.3)

# ── Panel derecho: puntos con etiqueta de texto (solo índices visibles) ───────
ax2 = axes[1]
for i, categoria in enumerate(CATEGORIAS_UNICAS):
    mask_idx = [j for j, c in enumerate(CATEGORIAS_CORPUS) if c == categoria]
    for idx in mask_idx:
        ax2.scatter(embeddings_2d[idx, 0], embeddings_2d[idx, 1],
                    c=[COLORES[i]], s=60, alpha=0.7)
        # Muestra solo las primeras 3 palabras del texto para no saturar
        etiqueta = " ".join(TEXTOS_CORPUS[idx].split()[:3]) + "..."
        ax2.annotate(etiqueta, (embeddings_2d[idx, 0], embeddings_2d[idx, 1]),
                     fontsize=5, alpha=0.7,
                     xytext=(3, 3), textcoords="offset points")

ax2.set_title("Con etiquetas de texto", fontsize=12)
ax2.set_xlabel(f"PC1 ({varianza[0]:.1%} var.)")
ax2.set_ylabel(f"PC2 ({varianza[1]:.1%} var.)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("embeddings_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado como embeddings_pca.png")

**Observaciones** (edita esta celda):

- ¿Se separan bien las categorías en 2D? ¿Cuáles se solapan más?
- La varianza total explicada es del ...%. ¿Es mucha o poca información la que se pierde?
- ¿Qué categorías están más cercanas entre sí? ¿Tiene sentido intuitivo?
- Si usaras 3 dimensiones en lugar de 2, ¿esperarías que las categorías se separasen más?

---
## Celda 4 — Comparativa: semántico vs palabras clave

Implementa una búsqueda por palabras clave básica (TF simple) y compárala
con la búsqueda semántica sobre las mismas queries.

Verás exactamente **cuándo falla cada enfoque** y por qué la semana 8
(búsqueda híbrida) combina ambos.

In [ ]:
import re
from collections import Counter

# ── Buscador por palabras clave (TF simple) ───────────────────────────────────

STOPWORDS_ES = {
    "el", "la", "los", "las", "un", "una", "unos", "unas", "de", "del",
    "en", "con", "para", "por", "que", "es", "son", "se", "su", "al",
    "a", "y", "o", "no", "más", "como", "pero", "si", "este", "esta",
    "esto", "ese", "esa", "eso", "cada", "muy", "bien", "está", "son",
}

def tokenizar(texto: str) -> list[str]:
    """Tokenización simple: minúsculas, solo letras y números."""
    tokens = re.findall(r"[a-záéíóúüñ0-9]+", texto.lower())
    return [t for t in tokens if t not in STOPWORDS_ES and len(t) > 2]

def buscar_palabras_clave(
    query: str,
    top_k: int = 5
) -> list[dict]:
    """Búsqueda por conteo de palabras en común (TF simple, sin IDF)."""
    tokens_query = set(tokenizar(query))
    if not tokens_query:
        return []

    scores = []
    for i, texto in enumerate(TEXTOS_CORPUS):
        tokens_doc = set(tokenizar(texto))
        coincidencias = tokens_query & tokens_doc
        # Score: palabras en común / total palabras de la query
        score = len(coincidencias) / len(tokens_query)
        if score > 0:
            scores.append({
                "score":         score,
                "texto":         texto,
                "categoria":     CATEGORIAS_CORPUS[i],
                "coincidencias": list(coincidencias)
            })

    scores.sort(key=lambda x: -x["score"])
    return scores[:top_k]


# ── Comparativa en 6 queries diseñadas para mostrar las diferencias ───────────

QUERIES_COMPARATIVA = [
    {
        "query":      "aprendizaje automático con supervisión",
        "descripcion": "Caso favorable a semántico (sinónimos)"
    },
    {
        "query":      "overfitting regularización L2",
        "descripcion": "Caso favorable a palabras clave (términos técnicos exactos)"
    },
    {
        "query":      "can ladrar",
        "descripcion": "Semántico encuentra 'perro', palabras clave no"
    },
    {
        "query":      "RGPD datos personales",
        "descripcion": "Acrónimo — ¿cómo lo maneja cada uno?"
    },
    {
        "query":      "ganar dinero trabajando solo",
        "descripcion": "Metáfora — semántico debería encontrar freelance"
    },
    {
        "query":      "Python async await coroutine",
        "descripcion": "Términos técnicos en inglés en corpus en español"
    },
]

for caso in QUERIES_COMPARATIVA:
    query = caso["query"]
    print(f"\n{'='*70}")
    print(f"Query:       '{query}'")
    print(f"Descripción: {caso['descripcion']}")

    res_semantico = buscar_semantico(query, top_k=2)
    res_keywords  = buscar_palabras_clave(query, top_k=2)

    print(f"\n  [SEMÁNTICO]")
    if res_semantico:
        for r in res_semantico:
            print(f"    {r['score']:.4f} | {r['texto'][:60]}...")
    else:
        print("    (sin resultados sobre umbral)")

    print(f"\n  [PALABRAS CLAVE]")
    if res_keywords:
        for r in res_keywords:
            coincide = ", ".join(r["coincidencias"])
            print(f"    {r['score']:.4f} | {r['texto'][:55]}... [{coincide}]")
    else:
        print("    (sin coincidencias exactas)")

print(f"\n{'='*70}")
print("Conclusión: ¿en qué casos cada método es superior?")

**Tabla comparativa** (completa con tus observaciones):

| Caso | Ganador | Por qué |
|------|---------|--------|
| sinónimos | Semántico | ... |
| términos técnicos exactos | Palabras clave | ... |
| can/perro | ... | ... |
| acrónimos | ... | ... |
| metáforas | ... | ... |
| inglés en corpus español | ... | ... |

**Conclusión:** ¿por qué la búsqueda híbrida (semana 8) es la solución correcta?

---
## Reto final — FAQ semántica

Construye un sistema de FAQ semántica para una empresa ficticia de software.

Dado un banco de 20 preguntas frecuentes con sus respuestas, cuando el usuario
formula una pregunta (con sus propias palabras, no necesariamente exacta),
el sistema encuentra la FAQ más parecida y devuelve su respuesta.

Esto es exactamente el caso de uso más simple de RAG: recuperar → devolver.

In [ ]:
# ── Banco de FAQs ─────────────────────────────────────────────────────────────

FAQS = [
    {"pregunta": "¿Cuánto cuesta el plan básico?",
     "respuesta": "El plan básico cuesta 29€/mes e incluye hasta 3 usuarios y 10 GB de almacenamiento."},
    {"pregunta": "¿Puedo cancelar mi suscripción en cualquier momento?",
     "respuesta": "Sí, puedes cancelar cuando quieras desde Configuración > Suscripción. No hay penalizaciones."},
    {"pregunta": "¿Ofrecéis descuento para startups?",
     "respuesta": "Sí, startups de menos de 2 años con menos de 10 empleados obtienen un 40% de descuento el primer año."},
    {"pregunta": "¿Cómo reseteo mi contraseña?",
     "respuesta": "En la pantalla de login, haz clic en '¿Olvidaste tu contraseña?' e introduce tu email."},
    {"pregunta": "¿Dónde están alojados mis datos?",
     "respuesta": "Los datos se almacenan en servidores dentro de la UE (Frankfurt, Irlanda), cumpliendo con el RGPD."},
    {"pregunta": "¿Tiene la aplicación integración con Slack?",
     "respuesta": "Sí, la integración con Slack está disponible en el plan Profesional y superior."},
    {"pregunta": "¿Cuántos usuarios puedo añadir?",
     "respuesta": "Básico: 3 usuarios. Profesional: 20 usuarios. Enterprise: ilimitados."},
    {"pregunta": "¿Hay versión móvil?",
     "respuesta": "Sí, tenemos apps nativas para iOS y Android disponibles en sus respectivas tiendas."},
    {"pregunta": "¿Cómo contacto con soporte técnico?",
     "respuesta": "Abre un ticket en soporte.miapp.com o escríbenos a soporte@miapp.com. Respondemos en menos de 4 horas."},
    {"pregunta": "¿Puedo exportar mis datos?",
     "respuesta": "Sí, puedes exportar todos tus datos en CSV o JSON desde Configuración > Exportar datos."},
    {"pregunta": "¿Qué métodos de pago aceptáis?",
     "respuesta": "Aceptamos tarjeta de crédito/débito (Visa, Mastercard), PayPal y transferencia bancaria para planes Enterprise."},
    {"pregunta": "¿Hay periodo de prueba gratuito?",
     "respuesta": "Sí, 14 días de prueba gratuita sin necesidad de tarjeta de crédito."},
    {"pregunta": "¿La aplicación funciona sin conexión a internet?",
     "respuesta": "Hay modo offline limitado en la app móvil. Para funcionalidad completa se requiere conexión."},
    {"pregunta": "¿Cómo añado un nuevo miembro al equipo?",
     "respuesta": "Ve a Configuración > Equipo > Invitar usuario. Introduce el email y asigna un rol."},
    {"pregunta": "¿Tenéis API pública?",
     "respuesta": "Sí, la API REST está disponible en el plan Profesional. Documentación en api.miapp.com."},
    {"pregunta": "¿Cómo cambio el método de pago?",
     "respuesta": "En Configuración > Facturación > Método de pago puedes actualizar o cambiar tu tarjeta."},
    {"pregunta": "¿La plataforma cumple con el RGPD?",
     "respuesta": "Sí, somos completamente conformes con el RGPD. Disponemos de DPA, registro de actividades y herramientas de consentimiento."},
    {"pregunta": "¿Puedo usar mi propio dominio?",
     "respuesta": "Sí, en el plan Enterprise puedes configurar un dominio personalizado (ej: app.tuempresa.com)."},
    {"pregunta": "¿Qué ocurre si supero el límite de almacenamiento?",
     "respuesta": "Recibirás un aviso al 80% y al 95%. Al llegar al 100% no podrás subir más archivos hasta ampliar o limpiar."},
    {"pregunta": "¿Ofrecéis formación o onboarding?",
     "respuesta": "Sí, ofrecemos webinars semanales gratuitos, tutoriales en video y onboarding personalizado para planes Enterprise."},
]


# ── Paso 1: indexa las preguntas de la FAQ ────────────────────────────────────
# TODO: genera los embeddings de todas las preguntas del FAQ
# Tip: usa generar_embeddings_batch() con la lista de preguntas

preguntas_faq  = [faq["pregunta"]  for faq in FAQS]
respuestas_faq = [faq["respuesta"] for faq in FAQS]

print("Indexando FAQs...")
# embeddings_faq = ...   # TODO
# print(f"FAQs indexadas: {embeddings_faq.shape}")


# ── Paso 2: implementa el buscador de FAQ ─────────────────────────────────────

def responder_faq(
    pregunta_usuario: str,
    umbral_confianza: float = 0.75
) -> dict:
    """
    Encuentra la FAQ más similar a la pregunta del usuario.

    Devuelve:
    {
        "respondida":  bool,
        "confianza":   float,
        "faq_encontrada": str,   # la pregunta del FAQ que coincidió
        "respuesta":   str,      # la respuesta de esa FAQ
        "mensaje":     str        # mensaje para el usuario
    }

    Si la similitud < umbral_confianza, no responde y sugiere contactar soporte.
    """
    # TODO: implementa la función
    # 1. Genera el embedding de la pregunta del usuario
    # 2. Calcula similitudes contra embeddings_faq
    # 3. Encuentra el índice con mayor similitud
    # 4. Si score >= umbral_confianza: devuelve la respuesta
    # 5. Si score < umbral_confianza: devuelve mensaje de no encontrado
    raise NotImplementedError("Implementa esta función")


# ── Paso 3: prueba con preguntas formuladas de forma distinta a las del FAQ ────
# Estas preguntas son distintas a las del FAQ pero deben recuperar la correcta

PREGUNTAS_TEST = [
    ("¿Cuánto hay que pagar cada mes por el plan más barato?",         "precio plan básico"),
    ("Quiero darme de baja, ¿cómo lo hago?",                           "cancelar suscripción"),
    ("No recuerdo mi clave de acceso",                                  "resetear contraseña"),
    ("¿Mis datos están en Europa?",                                    "ubicación datos / RGPD"),
    ("¿Funciona en el teléfono?",                                      "versión móvil"),
    ("¿Cuánto tiempo para probarlo sin pagar?",                        "periodo de prueba"),
    ("¿Tienen descuento para empresas nuevas?",                        "descuento startups"),
    ("¿Puedo conectarlo con otras herramientas mediante programación?", "API pública"),
    ("El sistema de archivos está casi lleno, ¿qué pasa?",             "límite almacenamiento"),
    ("¿Dónde está la normativa de privacidad?",                        "cumplimiento RGPD"),
    ("¿Puedo usar la app cuando no tengo wifi?",                       "modo offline"),
    ("¿Tienes soporte para empresas grandes?",                         "no debería coincidir directamente"),
]

# Descomenta cuando hayas implementado responder_faq()
# print("=" * 70)
# correctas = 0
# for pregunta, descripcion in PREGUNTAS_TEST:
#     resultado = responder_faq(pregunta)
#     icono = "✓" if resultado["respondida"] else "~"
#     print(f"\n{icono} [{descripcion}]")
#     print(f"  Query:     {pregunta}")
#     print(f"  FAQ match: {resultado.get('faq_encontrada', 'N/A')[:60]}...")
#     print(f"  Confianza: {resultado['confianza']:.4f}")
#     if resultado["respondida"]:
#         print(f"  Respuesta: {resultado['respuesta'][:80]}...")
#         correctas += 1
# 
# print(f"\n{'='*70}")
# print(f"Recuperadas: {correctas}/{len(PREGUNTAS_TEST)} ({correctas/len(PREGUNTAS_TEST):.0%})")

### Extensiones opcionales

1. **Caché de embeddings:** guarda los embeddings del corpus en un archivo `.npy` para no regenerarlos en cada ejecución.

2. **Top-3 con confianza:** en lugar de devolver solo el mejor resultado, devuelve los 3 más similares cuando la confianza del primero está entre 0.65 y 0.80, y deja que el usuario elija.

3. **UMAP en lugar de PCA:** instala `umap-learn` y repite la visualización de la celda 3 con UMAP. ¿Se separan mejor los clusters?

In [ ]:
# Extensión 1: caché de embeddings a disco
import numpy as np
from pathlib import Path

def guardar_embeddings(embeddings: np.ndarray, ruta: str) -> None:
    """Guarda embeddings en formato .npy para reutilizarlos."""
    np.save(ruta, embeddings)
    print(f"Embeddings guardados en {ruta} ({embeddings.shape})")

def cargar_o_generar_embeddings(textos: list[str], ruta_cache: str) -> np.ndarray:
    """Carga desde caché si existe; si no, genera y guarda."""
    if Path(ruta_cache).exists():
        embeddings = np.load(ruta_cache)
        print(f"Embeddings cargados desde caché: {embeddings.shape}")
        return embeddings
    else:
        print("Caché no encontrada. Generando embeddings...")
        embeddings = generar_embeddings_batch(textos)
        guardar_embeddings(embeddings, ruta_cache)
        return embeddings

# Usa la caché para el corpus del reto
# embeddings_faq_cacheado = cargar_o_generar_embeddings(preguntas_faq, "faq_embeddings.npy")

---
## Autoevaluación de la semana

1. **¿Qué funcionó bien?**  
   ...

2. **¿Qué no entendiste del todo?**  
   ...

3. **¿Qué cambiarías la semana que viene?**  
   ...

4. **Tiempo real dedicado:**  
   ...

---
*Notebook 05 — Semana 5 del Plan de Formación AI Engineering*